# 📦 Backup เอกสาร สส.5/18 จาก กกต. → Google Drive\n\n**วิธีใช้:**\n1. เปิด notebook นี้ใน Google Colab\n2. รัน Cell 1: Mount Google Drive\n3. รัน Cell 2: ตั้งค่า API Key + Mapping\n4. รัน Cell 3: เลือกจังหวัดที่ต้องการ\n5. รัน Cell 4: เริ่ม Backup!\n\n> ℹ️ ไฟล์จะถูกเซฟลง Google Drive ของคุณโดยตรง ที่ `MyDrive/สส.5_18_ข้อมูลเลือกตั้ง_2569/{จังหวัด}/`

In [ ]:
# Cell 1: Mount Google Drive + Authenticate
from google.colab import drive, auth
drive.mount('/content/drive')

# OAuth2 authentication (จะมี popup ให้ login + allow)
auth.authenticate_user()

from googleapiclient.discovery import build
from google.auth import default

creds, _ = default()
drive_service = build('drive', 'v3', credentials=creds)

print("✅ Google Drive mounted + authenticated!")

In [ ]:
# Cell 2: ตั้งค่า + ECT Drive Mapping (verified 2026-02-27)
import json, os, time, re
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

DRIVE_ROOT = "/content/drive/MyDrive/สส.5_18_ข้อมูลเลือกตั้ง_2569"

# === Verified folder IDs from ect_drive_mapping_verified.json ===
ECT_MAPPING = {
    "กำแพงเพชร": "1YFrEvow3-HwkcosJuXNeI82DL1WSrK_S",
    "แพร่": "1wD2ICNJHLhW0UaisZpW8ie49RC8Ba50v",
    "ลำปาง": "153yUnWv_2EWXSAbtsTTBE-wQYn9-6yXA",
    "เชียงใหม่": "1RWvYL-2KyyyCKGjF6qj39qKkpUjI6oML",
    "พิษณุโลก": "1I28S5sE3JOsbux5zk3oXp2gEe58-TpQ5",
    "ลำพูน": "1nTVc_RgxSJIAHN2Zk3Y6KEbCteobL_bH",
    "เชียงราย": "1w6ExwHN1c2qakXqUxAh7KmTJmZT-SSOo",
    "พิจิตร": "1At01M8EipiffkqpTRT2uln5FTcA-Mktj",
    "สุโขทัย": "1vDNUdsIBQUVVe1wX2sRfs-hzJ6g_eAkn",
    "น่าน": "1Qxfkwmm98At95sWBoMStaYaUvGNh6jxd",
    "เพชรบูรณ์": "11xsNcsLAmjIIpBZu48FdjBFMerqQjNAQ",
    "อุตรดิตถ์": "1GBBRBIh-EmpHNwRXIoL8NYeftv6a-Zf2",
    "พะเยา": "1e_3_iZiijJIUxY20L55tZYW4AAXTnezQ",
    "แม่ฮ่องสอน": "1fPkBLx7rwoK9f6QRl6gwFxLzXYYpPSb8",
    "กรุงเทพมหานคร": "1C1bvoAzR55wuJ6Dkt5OYWGz4ukv-F5Sl",
    "ปทุมธานี": "1bTPhst0RbMD-U9mFdxqOMzEyvOisQL6i",
    "สิงห์บุรี": "1SXJtGFdXWj5vptFftBaXYB0DRLKpRz2J",
    "ชัยนาท": "1UUm2NLLCM-2hwZ0-GyJ5FqP4-Kk2tBnx",
    "พระนครศรีอยุธยา": "11kxwA_elo8-IzYgFyyxplpFymLlqz3eY",
    "สุพรรณบุรี": "1t5GO6gg4W6rMGcO8pQOLYKLraVRUgAgR",
    "นครนายก": "1P_419iJaH_wsCuKhOJXf0BpbbZ8CCxNG",
    "ลพบุรี": "1UdK2NnBS1b_g_F4NciQuCgZHAZusI370",
    "สระบุรี": "1kjiOI6K37nvx8zDC0TRX-97JyOGY6SSu",
    "นครปฐม": "1jPhEkJQc0icPWRl29xwxGx9koXwKO1A1",
    "สมุทรปราการ": "15RY4_RDyMhgQWWmRANLg-Shh66L3Qq83",
    "อ่างทอง": "1pIq99HvFS-9AZ9MT1dWr6qV1kZG_bAmc",
    "นนทบุรี": "1MApGQ8YpAG1hVMOWqfKdfag3KLWYYWY5",
    "สมุทรสงคราม": "1KLHjx-aovF3TTcFJdzzrt-nQ7-wWuzAc",
    "อุทัยธานี": "1AEsdI9srCecTof5DeIAs9vYUay4KSzVH",
    "นครสวรรค์": "1EE6JUZARf71n4j9UZyIkf82TquHgWUe0",
    "สมุทรสาคร": "1UJ_DgxRSWCcGzqsIQxA_vWgZqsZNncD1",
    "กาฬสินธุ์": "1nRvDR4BWnW0j4DmeS4owe6Pb0iiUrudv",
    "มหาสารคาม": "1OLW1YTKxj4kTnxHYf0EdwZGKcE5Seojz",
    "ศรีสะเกษ": "1AZkRyP4_BwPOE7gd-_ExYvHZ2V_jwh9U",
    "ขอนแก่น": "1A89Hil8Yu_cFlU-FPY6FLfevaqOEg5IC",
    "มุกดาหาร": "17r8Me6sljlN1NC7er3skOZfpbV1w4aE0",
    "หนองคาย": "1pgb4CNpWnsS7t487PHavovUYYmO0VzlJ",
    "ชัยภูมิ": "11OAVEt8A7SfQLv9uPi9O-gLk37i_3CSI",
    "ยโสธร": "1eMX1g22UePTyLFRS3-GYQ-HSRyDC-_tm",
    "หนองบัวลำภู": "1JreB6tOF2w_8QkYActCc4182GBaEUH2u",
    "ร้อยเอ็ด": "1IRJHtBr1ohWfrotTcTbM_Ta39vSlkLkD",
    "อำนาจเจริญ": "14wkHdjHrTufhVFGsSY6rGT8Pst9HnjVu",
    "นครราชสีมา": "1VUAH6skupn_Y1Kbfm5C6WYQSmSqsESNf",
    "เลย": "1_MBbOA5r_HG5PA8dY8tzckP0nEzPPZcs",
    "อุดรธานี": "1picxYRg1bxW0QJyrTGG9zEV762sY--Kz",
    "บึงกาฬ": "1KR8FtPU5ZSlQvIS-YAAdYzHALbYb-0fr",
    "สกลนคร": "1EAH4dIYh2hgF0xXVTWNsRggtZS_2zaCD",
    "อุบลราชธานี": "1Nb7hJtDRoQy8VVR_cOfYhF7BHidRcuIH",
    "บุรีรัมย์": "1oPGf30Fo3_wELh5PpQibP24SXTVqga3q",
    "สุรินทร์": "1CYvOxPmnWfImGLJyv5Nktx7R6uR7VV6j",
    "จันทบุรี": "15oWc6hB8XVJ4ku-fsCUQl7DI0bBOzl-a",
    "ตราด": "1XLFjavED1BW_AG4xBDh5BfI6VQHv30_x",
    "สระแก้ว": "1JHP61jJf4ivKliXvS5kqZsAPz93lwujk",
    "ปราจีนบุรี": "1GnQ8lFpzfnpuy83KItOp5X5uqYsz1Wit",
    "ชลบุรี": "1tkp9vLv2nUSlECM4e9WnobfBr8RyTBzT",
    "ระยอง": "1uBC1GjNS_kmgemnO8HPZHO43oeNu1t1U",
    "กาญจนบุรี": "1h98dubKjCGNWqajx-KZx-bXlACzg6B4X",
    "ประจวบคีรีขันธ์": "1pV61BgH_CEOM-ETOvssJcYJBrjsiNuuV",
    "ราชบุรี": "1cbnTDoiRs1_BB60kx2nNlbUKkByOfNPI",
    "ตาก": "19vRSFjNCdHx2SqCb1fpWqC1UguoTmCNx",
    "เพชรบุรี": "1rLzQX8s86Dy5-gkSoaMr-fGfHOfyKOMj",
    "กระบี่": "1jG2TcSJVrmvOacW0kMaDaVzrU2wM6Quc",
    "ปัตตานี": "1_y3m72ukRQM1kTa5XPR_qiQC6foLci5a",
    "ระนอง": "1Zwqe61yZx6F2ZcBaf4HcYYPSifBAJJhN",
    "ชุมพร": "1Hhtc2g_Tr5cWt4mVeSKgx6GbP1ZpT85k",
    "พังงา": "1OsTJ5mpr5BGxC29w-hUDmJztZQonU6mZ",
    "สตูล": "1vSbfvVzsd6SCO2gIfw-AHuyJZqo9vovE",
    "ตรัง": "1bQiDrKRbJ9seMVA6XlUhRFUw5sRMovrz",
    "พัทลุง": "1iZAbINx1SXfr50Ul5ljAc68Si7tDV9Sz",
    "สงขลา": "1smYf3sOszd3onwYx-q1JthQ5_6bDgAjs",
    "นครศรีธรรมราช": "1Sn2_mAtcns6Q-Rn8QjlYdIYqvbiNt7DB",
    "ภูเก็ต": "1AsZpIJhIrt1XEmF8NASilMML_qf4We-C",
    "สุราษฎร์ธานี": "17P3jpRHAFCOlZOoo_z_XQRwqt34Ocd9P",
    "นราธิวาส": "1XRZIqhC4n6JEYCnON868am2pP4vMByE1",
    "ยะลา": "1GWDOvgZI4njIepaFVQ-skYZ7LUxF-ylT",
    "นครพนม": "11VfP-Cst29ZKdRSS7kj9g7IgxMKdUyNK",
    "ฉะเชิงเทรา": "1lHFX-tgeHOqAOSPIrkKx8-uIr9RZFIUB",
}

print(f"✅ Config loaded: {len(ECT_MAPPING)} จังหวัด, root={DRIVE_ROOT}")

In [ ]:
# Cell 3: เลือกจังหวัด
# เปลี่ยน TARGET:
#   "all"           = ทุกจังหวัด (ตาม anomaly ranking)
#   "บุรีรัมย์"       = จังหวัดเดียว
#   "บุรีรัมย์,ตาก"   = หลายจังหวัด

TARGET = "all"  # @param {type:"string"}

RANKING_ORDER = [
    "บุรีรัมย์", "ตาก", "ชัยภูมิ", "เพชรบูรณ์", "เพชรบุรี",
    "นครราชสีมา", "อุบลราชธานี", "พะเยา", "นครสวรรค์", "อ่างทอง",
    "อุทัยธานี", "สุรินทร์", "เลย", "สุพรรณบุรี", "ระยอง",
    "ศรีสะเกษ", "เชียงใหม่", "แพร่", "เชียงราย", "นครศรีธรรมราช",
    "สระแก้ว", "ชัยนาท", "สุโขทัย", "ระนอง", "ขอนแก่น",
    "ร้อยเอ็ด", "กาฬสินธุ์", "สกลนคร", "นครพนม", "ลำปาง",
    "แม่ฮ่องสอน", "กาญจนบุรี", "สตูล", "นราธิวาส",
    "กรุงเทพมหานคร", "พิจิตร", "สมุทรปราการ", "หนองบัวลำภู", "ปัตตานี",
    "พระนครศรีอยุธยา", "ลพบุรี", "สระบุรี", "ชลบุรี", "จันทบุรี",
    "ตราด", "ฉะเชิงเทรา", "ปราจีนบุรี", "นครนายก", "สงขลา",
    "ยโสธร", "อำนาจเจริญ", "บึงกาฬ", "นนทบุรี", "อุดรธานี",
    "หนองคาย", "มหาสารคาม", "มุกดาหาร", "ลำพูน", "อุตรดิตถ์",
    "น่าน", "กำแพงเพชร", "พิษณุโลก", "ประจวบคีรีขันธ์", "ราชบุรี",
    "นครปฐม", "สมุทรสาคร", "สมุทรสงคราม", "ปทุมธานี", "กระบี่",
    "พังงา", "ภูเก็ต", "สิงห์บุรี", "ชุมพร", "ตรัง",
    "พัทลุง", "ยะลา", "สุราษฎร์ธานี",
]

if TARGET.lower() == "all":
    provinces = RANKING_ORDER
elif "," in TARGET:
    provinces = [p.strip() for p in TARGET.split(",")]
else:
    provinces = [TARGET.strip()]

for p in provinces:
    if p not in ECT_MAPPING:
        raise ValueError(f"❌ ไม่พบจังหวัด '{p}'")

print(f"🎯 จะ backup {len(provinces)} จังหวัด")
for i, p in enumerate(provinces[:10], 1):
    prov_dir = os.path.join(DRIVE_ROOT, p)
    existing = len(list(Path(prov_dir).rglob("*.pdf"))) if os.path.exists(prov_dir) else 0
    print(f"   {i}. {p} ({existing} PDFs อยู่แล้ว)")
if len(provinces) > 10:
    print(f"   ... อีก {len(provinces)-10} จังหวัด")

In [ ]:
# Cell 4: (ข้ามได้ — ย้ายไปรวมใน Cell 5 แล้ว)
print("⏭️ ข้าม cell นี้ — ไปรัน Cell 5 เลย")

In [ ]:
# Cell 5: 🚀 ALL-IN-ONE Backup
# ต้องรัน Cell 1 (auth), Cell 2 (mapping), Cell 3 (provinces) ก่อน

import io, os, json, time, threading, re
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from google.auth import default as google_auth_default
import logging
logging.getLogger('google_auth_httplib2').setLevel(logging.ERROR)

print("🔧 สร้าง Drive service...", flush=True)
_creds, _ = google_auth_default()
_main_service = build('drive', 'v3', credentials=_creds)
_test = _main_service.files().list(pageSize=1, fields="files(id)").execute()
print(f"✅ Drive API OK")

# ---- Performance tuning ----
MAX_WORKERS = 20          # 20 threads (Drive API ≈13 req/s, retry handles 429)
SCAN_CACHE_DIR = DRIVE_ROOT
MAX_RETRY_ROUNDS = 3

def _sanitize_filename(name):
    return re.sub(r'[/\\]', '_', name)

def _retry(func, max_retries=6, base_delay=2):
    for attempt in range(max_retries):
        try:
            return func()
        except Exception as e:
            err = str(e)
            if any(c in err for c in ["500","503","429","Internal","Rate","Broken","Reset","timed"]):
                if attempt < max_retries - 1:
                    delay = base_delay * (2 ** min(attempt, 4))
                    time.sleep(delay)
                    continue
            raise

def _list_folder(svc, folder_id):
    all_files, pt = [], None
    while True:
        def _do(token=pt):
            return svc.files().list(
                q=f"'{folder_id}' in parents and trashed = false",
                fields="nextPageToken, files(id, name, mimeType, size)",
                pageSize=1000, pageToken=token,
                supportsAllDrives=True, includeItemsFromAllDrives=True,
            ).execute()
        data = _retry(_do)
        all_files.extend(data.get("files", []))
        pt = data.get("nextPageToken")
        if not pt: break
    return all_files

def _walk_folder(svc, folder_id, path="", depth=0):
    results = []
    try:
        items = _list_folder(svc, folder_id)
    except Exception as e:
        print(f"      ⚠️ scan error at '{path}': {e}", flush=True)
        return results
    folders = [f for f in items if f["mimeType"] == "application/vnd.google-apps.folder"]
    pdfs = [f for f in items if f.get("name","").lower().endswith(".pdf")]
    if pdfs and depth <= 2:
        print(f"      📁 {path or '(root)'} → {len(pdfs)} PDFs", flush=True)
    for pdf in pdfs:
        results.append({"path": path, "file": pdf})
    for folder in sorted(folders, key=lambda f: f["name"]):
        sub = f"{path}/{folder['name']}" if path else folder["name"]
        results.extend(_walk_folder(svc, folder["id"], sub, depth+1))
        time.sleep(0.05)
    return results

def _save_cache(province, items):
    os.makedirs(SCAN_CACHE_DIR, exist_ok=True)
    p = os.path.join(SCAN_CACHE_DIR, f"_scan_cache_{province}.json")
    with open(p, "w", encoding="utf-8") as f:
        json.dump(items, f, ensure_ascii=False)
    print(f"   💾 Cache saved: {len(items)} items")

def _load_cache(province):
    p = os.path.join(SCAN_CACHE_DIR, f"_scan_cache_{province}.json")
    if os.path.exists(p):
        with open(p, "r", encoding="utf-8") as f:
            items = json.load(f)
        print(f"   📦 Cache loaded: {len(items)} items (skip scan!)")
        return items
    return None

_tl = threading.local()
def _get_svc():
    if not hasattr(_tl, 'svc'):
        c, _ = google_auth_default()
        _tl.svc = build('drive', 'v3', credentials=c)
    return _tl.svc

def _download_one(item, prov_dir):
    pdf = item["file"]
    fname = _sanitize_filename(pdf["name"])
    target_dir = os.path.join(prov_dir, item["path"]) if item["path"] else prov_dir
    os.makedirs(target_dir, exist_ok=True)
    dest = os.path.join(target_dir, fname)
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        return ("skip", fname, os.path.getsize(dest))
    svc = _get_svc()
    for attempt in range(4):
        try:
            req = svc.files().get_media(fileId=pdf["id"], supportsAllDrives=True)
            buf = io.BytesIO()
            dl = MediaIoBaseDownload(buf, req)
            done = False
            while not done:
                _, done = dl.next_chunk()
            with open(dest, "wb") as f:
                f.write(buf.getvalue())
            return ("ok", fname, os.path.getsize(dest))
        except Exception as e:
            err = str(e)
            if "403" in err or "forbidden" in err.lower():
                return ("403", fname, 0)
            if any(c in err for c in ["429","500","503","Rate","Internal","Reset","timed"]):
                if attempt < 3:
                    time.sleep(2 * (2 ** attempt))
                    continue
            if attempt == 3:
                return ("fail", fname, err[:80])
            time.sleep(2*(attempt+1))

class Counter:
    def __init__(self, total):
        self.total = total
        self.ok = self.skip = self.r403 = self.fail = self.mb = 0
        self.fail_items = []
        self.lock = threading.Lock()
        self.t0 = time.time()
    def add(self, st, fn, sz, item=None):
        with self.lock:
            if st == "ok":
                self.ok += 1; self.mb += sz/1024/1024
                n = self.ok + self.skip + self.r403 + self.fail
                if self.ok <= 3 or self.ok % 50 == 0 or n == self.total:
                    rate = self.ok / max(time.time()-self.t0, 1) * 60
                    print(f"   ✅ [{n}/{self.total}] {self.mb:.0f}MB ~{rate:.0f}/min", flush=True)
            elif st == "skip": self.skip += 1
            elif st == "403":
                self.r403 += 1
                if self.r403 <= 2: print(f"   ⛔ 403: {fn}", flush=True)
                elif self.r403 == 3: print(f"   ⛔ (ซ่อน 403 ที่เหลือ)", flush=True)
            else:
                self.fail += 1
                if item: self.fail_items.append(item)
                if self.fail <= 5: print(f"   ❌ {fn}: {sz}", flush=True)
                elif self.fail == 6: print(f"   ❌ (ซ่อน error — จะ retry อัตโนมัติ)", flush=True)

def _run_downloads(items, prov_dir):
    ctr = Counter(len(items))
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = {ex.submit(_download_one, it, prov_dir): it for it in items}
        for fut in as_completed(futs):
            it = futs[fut]
            try:
                st, fn, sz = fut.result()
                ctr.add(st, fn, sz, it)
            except Exception as e:
                ctr.add("fail", it["file"]["name"], str(e)[:80], it)
    return ctr

# ============================================================
# MAIN LOOP
# ============================================================
all_results = []
t_start = time.time()

for pi, province in enumerate(provinces, 1):
    fid = ECT_MAPPING[province]
    pdir = os.path.join(DRIVE_ROOT, province)
    os.makedirs(pdir, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"📍 [{pi}/{len(provinces)}] {province}")

    pdf_items = _load_cache(province)
    if pdf_items is None:
        print(f"   🔍 สแกน folder...", flush=True)
        t0 = time.time()
        pdf_items = _walk_folder(_main_service, fid)
        if pdf_items:
            _save_cache(province, pdf_items)
        print(f"   📄 พบ {len(pdf_items)} PDFs ({time.time()-t0:.0f}s)")

    if not pdf_items:
        print(f"   ⚠️ ไม่พบ PDF (folder อาจถูกลบหรือไม่มีสิทธิ์)")
        all_results.append({"province": province, "ok": False, "status": "empty"})
        continue

    print(f"   🚀 Download ({MAX_WORKERS} threads)...", flush=True)
    ctr = _run_downloads(pdf_items, pdir)

    for retry_round in range(1, MAX_RETRY_ROUNDS + 1):
        if not ctr.fail_items: break
        retry_items = list(ctr.fail_items)
        print(f"\n   🔄 Retry {retry_round}/{MAX_RETRY_ROUNDS}: {len(retry_items)} ไฟล์", flush=True)
        time.sleep(5)
        retry_ctr = _run_downloads(retry_items, pdir)
        ctr.ok += retry_ctr.ok
        ctr.mb += retry_ctr.mb
        ctr.r403 += retry_ctr.r403
        ctr.fail = retry_ctr.fail
        ctr.fail_items = retry_ctr.fail_items

    actual = len(list(Path(pdir).rglob("*.pdf")))
    accessible = len(pdf_items) - ctr.r403
    pct = round(actual/accessible*100, 1) if accessible > 0 else 100
    icon = "✅" if actual >= accessible else "⚠️"
    print(f"\n   📊 {icon} {province}: {actual}/{accessible} ({pct}%) | 403:{ctr.r403} | fail:{ctr.fail} | ⏱️{time.time()-ctr.t0:.0f}s")

    all_results.append({
        "province": province, "ok": actual >= accessible, "total": len(pdf_items),
        "accessible": accessible, "actual": actual, "pct": pct,
        "dl": ctr.ok, "skip": ctr.skip, "r403": ctr.r403, "fail": ctr.fail
    })

el = time.time() - t_start
print(f"\n{'='*60}")
print(f"🏁 Backup เสร็จ! ({int(el//3600)}h {int(el%3600//60)}m {int(el%60)}s)")
print(f"{'='*60}")
total_dl = total_actual = total_accessible = 0
for r in all_results:
    i = "✅" if r.get("ok") else "⚠️"
    print(f"  {i} {r['province']}: {r.get('actual','?')}/{r.get('accessible','?')} ({r.get('pct','?')}%)")
    total_dl += r.get("dl", 0)
    total_actual += r.get("actual", 0)
    total_accessible += r.get("accessible", 0)
pct = round(total_actual/total_accessible*100,1) if total_accessible > 0 else 0
print(f"\n📊 รวม: {total_actual}/{total_accessible} PDFs ({pct}%) | ดาวน์โหลดใหม่: {total_dl}")
print(f"📁 {DRIVE_ROOT}")

In [ ]:
# Cell 6: 🔄 RETRY จังหวัดที่ไม่ครบ 100% (v7 — เพิ่ม นครพนม + ฉะเชิงเทรา)
# ⚡ Self-contained — ต้องรันแค่ Cell 1 (auth) + Cell 2 (mapping) ก่อน
# ข้าม Cell 3, 4, 5 ได้เลย

import io, os, json, time, threading, re, hashlib, unicodedata
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from google.auth import default as google_auth_default
import logging
logging.getLogger('google_auth_httplib2').setLevel(logging.ERROR)

print("🔧 สร้าง Drive service...", flush=True)
_creds, _ = google_auth_default()
_main_service = build('drive', 'v3', credentials=_creds)
_test = _main_service.files().list(pageSize=1, fields="files(id)").execute()
print(f"✅ Drive API OK")

MAX_WORKERS = 20
SCAN_CACHE_DIR = DRIVE_ROOT
MAX_RETRY_ROUNDS_V2 = 5

# ✅ 2 จังหวัดที่เหลือ — folder ID ใหม่ (แก้แล้วใน Cell 2)
RETRY_PROVINCES = [
    "นครพนม",
    "ฉะเชิงเทรา",
]

def _sanitize_filename(name):
    name = re.sub(r'[/\\]', '_', name)
    name_bytes = name.encode('utf-8')
    if len(name_bytes) > 200:
        base, ext = os.path.splitext(name)
        h = hashlib.md5(name.encode('utf-8')).hexdigest()[:8]
        max_base_bytes = 200 - len(ext.encode('utf-8')) - 9
        while len(base.encode('utf-8')) > max_base_bytes:
            base = base[:-1]
        name = f"{base}_{h}{ext}"
    return name

def _normalize_key(text):
    return unicodedata.normalize('NFC', text)

def _dedup_items(items):
    seen = set()
    unique = []
    dup_count = 0
    for item in items:
        fname = _sanitize_filename(item["file"]["name"])
        key = (_normalize_key(item.get("path", "")), _normalize_key(fname))
        if key not in seen:
            seen.add(key)
            unique.append(item)
        else:
            dup_count += 1
    if dup_count > 0:
        print(f"   🔄 Dedup: ลบ {dup_count} entries ซ้ำ ({len(items)} → {len(unique)})")
    return unique

def _retry(func, max_retries=6, base_delay=2):
    for attempt in range(max_retries):
        try:
            return func()
        except Exception as e:
            err = str(e)
            if any(c in err for c in ["500","503","429","Internal","Rate","Broken","Reset","timed"]):
                if attempt < max_retries - 1:
                    delay = base_delay * (2 ** min(attempt, 4))
                    time.sleep(delay)
                    continue
            raise

def _list_folder(svc, folder_id):
    all_files, pt = [], None
    while True:
        def _do(token=pt):
            return svc.files().list(
                q=f"'{folder_id}' in parents and trashed = false",
                fields="nextPageToken, files(id, name, mimeType, size)",
                pageSize=1000, pageToken=token,
                supportsAllDrives=True, includeItemsFromAllDrives=True,
            ).execute()
        data = _retry(_do)
        all_files.extend(data.get("files", []))
        pt = data.get("nextPageToken")
        if not pt: break
    return all_files

def _walk_folder(svc, folder_id, path="", depth=0):
    results = []
    try:
        items = _list_folder(svc, folder_id)
    except Exception as e:
        print(f"      ⚠️ scan error at '{path}': {e}", flush=True)
        return results
    folders = [f for f in items if f["mimeType"] == "application/vnd.google-apps.folder"]
    pdfs = [f for f in items if f.get("name","").lower().endswith(".pdf")]
    if pdfs and depth <= 2:
        print(f"      📁 {path or '(root)'} → {len(pdfs)} PDFs", flush=True)
    for pdf in pdfs:
        results.append({"path": path, "file": pdf})
    for folder in sorted(folders, key=lambda f: f["name"]):
        sub = f"{path}/{folder['name']}" if path else folder["name"]
        results.extend(_walk_folder(svc, folder["id"], sub, depth+1))
    return results

def _save_cache(province, items):
    os.makedirs(SCAN_CACHE_DIR, exist_ok=True)
    p = os.path.join(SCAN_CACHE_DIR, f"_scan_cache_{province}.json")
    with open(p, "w", encoding="utf-8") as f:
        json.dump(items, f, ensure_ascii=False)
    print(f"   💾 Cache saved: {len(items)} items")

def _load_cache(province):
    p = os.path.join(SCAN_CACHE_DIR, f"_scan_cache_{province}.json")
    if os.path.exists(p):
        with open(p, "r", encoding="utf-8") as f:
            items = json.load(f)
        print(f"   📦 Cache loaded: {len(items)} items")
        return items
    return None

_tl = threading.local()
def _get_svc():
    if not hasattr(_tl, 'svc'):
        c, _ = google_auth_default()
        _tl.svc = build('drive', 'v3', credentials=c)
    return _tl.svc

def _download_one(item, prov_dir):
    pdf = item["file"]
    fname = _sanitize_filename(pdf["name"])
    target_dir = os.path.join(prov_dir, item["path"]) if item["path"] else prov_dir
    os.makedirs(target_dir, exist_ok=True)
    dest = os.path.join(target_dir, fname)
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        return ("skip", fname, os.path.getsize(dest))
    nfc_fname = _normalize_key(fname)
    nfc_dest = os.path.join(target_dir, nfc_fname)
    if nfc_dest != dest and os.path.exists(nfc_dest) and os.path.getsize(nfc_dest) > 0:
        return ("skip", fname, os.path.getsize(nfc_dest))
    old_fname = re.sub(r'[/\\]', '_', pdf["name"])
    old_dest = os.path.join(target_dir, old_fname)
    if old_dest != dest and os.path.exists(old_dest) and os.path.getsize(old_dest) > 0:
        return ("skip", fname, os.path.getsize(old_dest))
    svc = _get_svc()
    for attempt in range(4):
        try:
            req = svc.files().get_media(fileId=pdf["id"], supportsAllDrives=True)
            buf = io.BytesIO()
            dl = MediaIoBaseDownload(buf, req)
            done = False
            while not done:
                _, done = dl.next_chunk()
            with open(dest, "wb") as f:
                f.write(buf.getvalue())
            return ("ok", fname, os.path.getsize(dest))
        except Exception as e:
            err = str(e)
            if "403" in err or "forbidden" in err.lower():
                return ("403", fname, 0)
            if any(c in err for c in ["429","500","503","Rate","Internal","Reset","timed"]):
                if attempt < 3:
                    time.sleep(2 * (2 ** attempt))
                    continue
            if attempt == 3:
                return ("fail", fname, err[:80])
            time.sleep(2*(attempt+1))

class Counter:
    def __init__(self, total):
        self.total = total
        self.ok = self.skip = self.r403 = self.fail = self.mb = 0
        self.fail_items = []
        self.lock = threading.Lock()
        self.t0 = time.time()
    def add(self, st, fn, sz, item=None):
        with self.lock:
            if st == "ok":
                self.ok += 1; self.mb += sz/1024/1024
                n = self.ok + self.skip + self.r403 + self.fail
                if self.ok <= 3 or self.ok % 50 == 0 or n == self.total:
                    rate = self.ok / max(time.time()-self.t0, 1) * 60
                    print(f"   ✅ [{n}/{self.total}] {self.mb:.0f}MB ~{rate:.0f}/min", flush=True)
            elif st == "skip": self.skip += 1
            elif st == "403":
                self.r403 += 1
                if self.r403 <= 2: print(f"   ⛔ 403: {fn}", flush=True)
                elif self.r403 == 3: print(f"   ⛔ (ซ่อน 403 ที่เหลือ)", flush=True)
            else:
                self.fail += 1
                if item: self.fail_items.append(item)
                if self.fail <= 5: print(f"   ❌ {fn}: {sz}", flush=True)
                elif self.fail == 6: print(f"   ❌ (ซ่อน error — จะ retry อัตโนมัติ)", flush=True)

def _run_downloads(items, prov_dir):
    ctr = Counter(len(items))
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = {ex.submit(_download_one, it, prov_dir): it for it in items}
        for fut in as_completed(futs):
            it = futs[fut]
            try:
                st, fn, sz = fut.result()
                ctr.add(st, fn, sz, it)
            except Exception as e:
                ctr.add("fail", it["file"]["name"], str(e)[:80], it)
    return ctr

# ============================================================
# BACKUP LOOP — นครพนม + ฉะเชิงเทรา (folder ID ใหม่)
# ============================================================
print(f"\n🚀 Backup {len(RETRY_PROVINCES)} จังหวัดที่เหลือ (v7 — folder ID ใหม่)")
print(f"{'='*60}")

all_results = []
t_start = time.time()

for pi, province in enumerate(RETRY_PROVINCES, 1):
    fid = ECT_MAPPING[province]
    pdir = os.path.join(DRIVE_ROOT, province)
    os.makedirs(pdir, exist_ok=True)

    existing_before = len(list(Path(pdir).rglob("*.pdf")))

    print(f"{'='*60}")
    print(f"📍 [{pi}/{len(RETRY_PROVINCES)}] {province} (มีอยู่: {existing_before} PDFs)")
    print(f"   📂 Folder: https://drive.google.com/drive/folders/{fid}")

    # ⚡ ลบ cache เดิม (ถ้ามี) เพราะ folder ID เปลี่ยน
    old_cache = os.path.join(SCAN_CACHE_DIR, f"_scan_cache_{province}.json")
    if os.path.exists(old_cache):
        os.remove(old_cache)
        print(f"   🗑️ ลบ cache เดิม (folder ID เปลี่ยน)")

    print(f"   🔍 สแกน folder ใหม่...", flush=True)
    t0 = time.time()
    pdf_items = _walk_folder(_main_service, fid)
    if pdf_items:
        _save_cache(province, pdf_items)
    print(f"   📄 พบ {len(pdf_items)} PDFs ({time.time()-t0:.0f}s)")

    if not pdf_items:
        print(f"   ⚠️ ไม่พบ PDF (folder อาจถูกลบหรือไม่มีสิทธิ์)")
        all_results.append({"province": province, "ok": False, "status": "empty"})
        continue

    pdf_items = _dedup_items(pdf_items)

    print(f"   🚀 Download ({MAX_WORKERS} threads)...", flush=True)
    ctr = _run_downloads(pdf_items, pdir)

    for retry_round in range(1, MAX_RETRY_ROUNDS_V2 + 1):
        if not ctr.fail_items:
            break
        retry_items = list(ctr.fail_items)
        wait = 5 * retry_round
        print(f"\n   🔄 Retry {retry_round}/{MAX_RETRY_ROUNDS_V2}: {len(retry_items)} ไฟล์ (wait {wait}s)", flush=True)
        time.sleep(wait)
        retry_ctr = _run_downloads(retry_items, pdir)
        ctr.ok += retry_ctr.ok
        ctr.mb += retry_ctr.mb
        ctr.r403 += retry_ctr.r403
        ctr.fail = retry_ctr.fail
        ctr.fail_items = retry_ctr.fail_items

    actual = len(list(Path(pdir).rglob("*.pdf")))
    new_files = actual - existing_before
    accessible = len(pdf_items) - ctr.r403

    is_complete = ctr.fail == 0
    dup_gap = accessible - actual if accessible > actual else 0

    if is_complete and dup_gap > 0:
        pct = 100.0
        display_total = actual
    else:
        pct = round(actual / accessible * 100, 1) if accessible > 0 else 100
        display_total = accessible

    icon = "✅" if is_complete else "⚠️"
    dup_note = f" (dup:{dup_gap} ข้าม)" if dup_gap > 0 else ""
    print(f"\n   📊 {icon} {province}: {actual}/{display_total} ({pct}%){dup_note} | +{new_files} new | 403:{ctr.r403} | fail:{ctr.fail}\n")

    all_results.append({
        "province": province, "ok": is_complete, "total": len(pdf_items),
        "accessible": accessible, "actual": actual, "pct": pct,
        "display_total": display_total,
        "dl": ctr.ok, "skip": ctr.skip, "r403": ctr.r403, "fail": ctr.fail,
        "new": new_files, "dup_gap": dup_gap,
    })

el = time.time() - t_start
print(f"\n{'='*60}")
print(f"🏁 Backup เสร็จ! ({int(el//3600)}h {int(el%3600//60)}m {int(el%60)}s)")
print(f"{'='*60}")
for r in all_results:
    i = "✅" if r.get("ok") else "⚠️"
    new_str = f" (+{r.get('new', 0)} new)" if r.get('new', 0) > 0 else ""
    dt = r.get('display_total', r.get('accessible', '?'))
    print(f"  {i} {r['province']}: {r.get('actual','?')}/{dt} ({r.get('pct','?')}%){new_str}")

ok_count = sum(1 for r in all_results if r.get("ok"))
if ok_count == len(RETRY_PROVINCES):
    print(f"\n🎉 ครบ 77/77 จังหวัดแล้ว!! 🎉")
else:
    still = [r['province'] for r in all_results if not r.get("ok")]
    print(f"\n⚠️ ยังไม่ครบ: {', '.join(still)}")
print(f"📁 {DRIVE_ROOT}")

In [ ]:
# Cell 7: 📊 FULL AUDIT — นับไฟล์จริงบน Drive ทั้ง 77 จังหวัด
# ⚡ รันแค่ Cell 1 (auth) + Cell 2 (mapping) ก่อน แล้วรัน cell นี้ได้เลย
# ไม่ดาวน์โหลดอะไร — แค่นับไฟล์ที่มีอยู่จริง

import os, json, time
from pathlib import Path

RANKING_ORDER = [
    "บุรีรัมย์", "ตาก", "ชัยภูมิ", "เพชรบูรณ์", "เพชรบุรี",
    "นครราชสีมา", "อุบลราชธานี", "พะเยา", "นครสวรรค์", "อ่างทอง",
    "อุทัยธานี", "สุรินทร์", "เลย", "สุพรรณบุรี", "ระยอง",
    "ศรีสะเกษ", "เชียงใหม่", "แพร่", "เชียงราย", "นครศรีธรรมราช",
    "สระแก้ว", "ชัยนาท", "สุโขทัย", "ระนอง", "ขอนแก่น",
    "ร้อยเอ็ด", "กาฬสินธุ์", "สกลนคร", "นครพนม", "ลำปาง",
    "แม่ฮ่องสอน", "กาญจนบุรี", "สตูล", "นราธิวาส",
    "กรุงเทพมหานคร", "พิจิตร", "สมุทรปราการ", "หนองบัวลำภู", "ปัตตานี",
    "พระนครศรีอยุธยา", "ลพบุรี", "สระบุรี", "ชลบุรี", "จันทบุรี",
    "ตราด", "ฉะเชิงเทรา", "ปราจีนบุรี", "นครนายก", "สงขลา",
    "ยโสธร", "อำนาจเจริญ", "บึงกาฬ", "นนทบุรี", "อุดรธานี",
    "หนองคาย", "มหาสารคาม", "มุกดาหาร", "ลำพูน", "อุตรดิตถ์",
    "น่าน", "กำแพงเพชร", "พิษณุโลก", "ประจวบคีรีขันธ์", "ราชบุรี",
    "นครปฐม", "สมุทรสาคร", "สมุทรสงคราม", "ปทุมธานี", "กระบี่",
    "พังงา", "ภูเก็ต", "สิงห์บุรี", "ชุมพร", "ตรัง",
    "พัทลุง", "ยะลา", "สุราษฎร์ธานี",
]

print(f"📊 FULL AUDIT — นับไฟล์จริงทั้ง {len(RANKING_ORDER)} จังหวัด")
print(f"📁 {DRIVE_ROOT}")
print(f"{'='*70}\n")

t0 = time.time()
results = []
grand_total = 0

for i, prov in enumerate(RANKING_ORDER, 1):
    pdir = os.path.join(DRIVE_ROOT, prov)

    if not os.path.isdir(pdir):
        results.append({"province": prov, "actual": 0, "status": "no_folder", "cached": 0})
        print(f"  {i:2d}. ❌ {prov}: ไม่พบโฟลเดอร์")
        continue

    actual = len(list(Path(pdir).rglob("*.pdf")))

    # อ่านจำนวนจาก cache (ถ้ามี)
    cache_path = os.path.join(DRIVE_ROOT, f"_scan_cache_{prov}.json")
    cached = 0
    if os.path.exists(cache_path):
        try:
            with open(cache_path, "r", encoding="utf-8") as f:
                cached = len(json.load(f))
        except:
            pass

    grand_total += actual
    icon = "✅" if actual > 0 else "⚠️"
    cache_note = f" (cache:{cached})" if cached > 0 and cached != actual else ""
    results.append({"province": prov, "actual": actual, "status": "ok", "cached": cached})
    print(f"  {i:2d}. {icon} {prov}: {actual:,} PDFs{cache_note}")

el = time.time() - t0

# สรุป
print(f"\n{'='*70}")
print(f"🏁 AUDIT เสร็จ ({el:.0f}s)\n")

ok_provinces = [r for r in results if r["status"] == "ok" and r["actual"] > 0]
zero_provinces = [r for r in results if r["status"] == "ok" and r["actual"] == 0]
no_folder = [r for r in results if r["status"] == "no_folder"]

print(f"📊 รวมทั้งหมด: {grand_total:,} PDFs จาก {len(ok_provinces)} จังหวัด")
print(f"   ✅ มีข้อมูลครบ: {len(ok_provinces)}/77 จังหวัด")

if zero_provinces:
    print(f"   ⚠️ มีโฟลเดอร์แต่ 0 ไฟล์: {len(zero_provinces)} → {', '.join(r['province'] for r in zero_provinces)}")
if no_folder:
    print(f"   ❌ ไม่พบโฟลเดอร์: {len(no_folder)} → {', '.join(r['province'] for r in no_folder)}")

# ตาราง Top 10 + Bottom 10
print(f"\n{'='*70}")
sorted_by_count = sorted(ok_provinces, key=lambda r: r["actual"], reverse=True)
print("📈 Top 10:")
for r in sorted_by_count[:10]:
    print(f"   {r['province']}: {r['actual']:,}")
print("\n📉 Bottom 10:")
for r in sorted_by_count[-10:]:
    print(f"   {r['province']}: {r['actual']:,}")

# สรุปสุดท้าย
print(f"\n{'='*70}")
coverage = len(ok_provinces) / 77 * 100
print(f"🎯 Coverage: {len(ok_provinces)}/77 จังหวัด ({coverage:.1f}%)")
print(f"📄 Total PDFs: {grand_total:,}")
print(f"📁 {DRIVE_ROOT}")
if len(ok_provinces) == 77:
    print(f"\n🎉🎉🎉 ครบ 77/77 จังหวัด!! ข้อมูลดิบครบถ้วน — พร้อมใช้เป็นหลักฐาน 🎉🎉🎉")
elif len(ok_provinces) >= 75:
    print(f"\n✅ ข้อมูลดิบครบถ้วนสำหรับ {len(ok_provinces)} จังหวัด — พร้อมใช้เป็นหลักฐาน")
    missing = [r['province'] for r in results if r["status"] != "ok" or r["actual"] == 0]
    if missing:
        print(f"⚠️ เหลือ: {', '.join(missing)}")

# Cell 7: 📊 FULL AUDIT — นับไฟล์จริงบน Drive ทั้ง 77 จังหวัด
# ⚡ รันแค่ Cell 1 (auth) + Cell 2 (mapping) ก่อน แล้วรัน cell นี้ได้เลย
# ไม่ดาวน์โหลดอะไร — แค่นับไฟล์ที่มีอยู่จริง

import os, json, time
from pathlib import Path

RANKING_ORDER = [
    "บุรีรัมย์", "ตาก", "ชัยภูมิ", "เพชรบูรณ์", "เพชรบุรี",
    "นครราชสีมา", "อุบลราชธานี", "พะเยา", "นครสวรรค์", "อ่างทอง",
    "อุทัยธานี", "สุรินทร์", "เลย", "สุพรรณบุรี", "ระยอง",
    "ศรีสะเกษ", "เชียงใหม่", "แพร่", "เชียงราย", "นครศรีธรรมราช",
    "สระแก้ว", "ชัยนาท", "สุโขทัย", "ระนอง", "ขอนแก่น",
    "ร้อยเอ็ด", "กาฬสินธุ์", "สกลนคร", "นครพนม", "ลำปาง",
    "แม่ฮ่องสอน", "กาญจนบุรี", "สตูล", "นราธิวาส",
    "กรุงเทพมหานคร", "พิจิตร", "สมุทรปราการ", "หนองบัวลำภู", "ปัตตานี",
    "พระนครศรีอยุธยา", "ลพบุรี", "สระบุรี", "ชลบุรี", "จันทบุรี",
    "ตราด", "ฉะเชิงเทรา", "ปราจีนบุรี", "นครนายก", "สงขลา",
    "ยโสธร", "อำนาจเจริญ", "บึงกาฬ", "นนทบุรี", "อุดรธานี",
    "หนองคาย", "มหาสารคาม", "มุกดาหาร", "ลำพูน", "อุตรดิตถ์",
    "น่าน", "กำแพงเพชร", "พิษณุโลก", "ประจวบคีรีขันธ์", "ราชบุรี",
    "นครปฐม", "สมุทรสาคร", "สมุทรสงคราม", "ปทุมธานี", "กระบี่",
    "พังงา", "ภูเก็ต", "สิงห์บุรี", "ชุมพร", "ตรัง",
    "พัทลุง", "ยะลา", "สุราษฎร์ธานี",
]

KNOWN_404 = {"นครพนม", "ฉะเชิงเทรา"}

print(f"📊 FULL AUDIT — นับไฟล์จริงทั้ง {len(RANKING_ORDER)} จังหวัด")
print(f"📁 {DRIVE_ROOT}")
print(f"{'='*70}\n")

t0 = time.time()
results = []
grand_total = 0

for i, prov in enumerate(RANKING_ORDER, 1):
    pdir = os.path.join(DRIVE_ROOT, prov)
    if prov in KNOWN_404:
        results.append({"province": prov, "actual": 0, "status": "404", "cached": 0})
        print(f"  {i:2d}. ⛔ {prov}: folder ID ผิด (404)")
        continue
    if not os.path.isdir(pdir):
        results.append({"province": prov, "actual": 0, "status": "no_folder", "cached": 0})
        print(f"  {i:2d}. ❌ {prov}: ไม่พบโฟลเดอร์")
        continue
    actual = len(list(Path(pdir).rglob("*.pdf")))
    # อ่านจำนวนจาก cache (ถ้ามี)
    cache_path = os.path.join(DRIVE_ROOT, f"_scan_cache_{prov}.json")
    cached = 0
    if os.path.exists(cache_path):
        try:
            with open(cache_path, "r", encoding="utf-8") as f:
                cached = len(json.load(f))
        except:
            pass
    grand_total += actual
    icon = "✅" if actual > 0 else "⚠️"
    cache_note = f" (cache:{cached})" if cached > 0 and cached != actual else ""
    results.append({"province": prov, "actual": actual, "status": "ok", "cached": cached})
    print(f"  {i:2d}. {icon} {prov}: {actual:,} PDFs{cache_note}")

el = time.time() - t0

# สรุป
print(f"\n{'='*70}")
print(f"🏁 AUDIT เสร็จ ({el:.0f}s)\n")

ok_provinces = [r for r in results if r["status"] == "ok" and r["actual"] > 0]
err_provinces = [r for r in results if r["status"] == "404"]
zero_provinces = [r for r in results if r["status"] == "ok" and r["actual"] == 0]
no_folder = [r for r in results if r["status"] == "no_folder"]

print(f"📊 รวมทั้งหมด: {grand_total:,} PDFs จาก {len(ok_provinces)} จังหวัด")
print(f"   ✅ มีข้อมูลครบ: {len(ok_provinces)}/77 จังหวัด")

if err_provinces:
    print(f"   ⛔ Folder ID ผิด (404): {len(err_provinces)} จังหวัด → {', '.join(r['province'] for r in err_provinces)}")
if zero_provinces:
    print(f"   ⚠️ มีโฟลเดอร์แต่ 0 ไฟล์: {len(zero_provinces)} → {', '.join(r['province'] for r in zero_provinces)}")
if no_folder:
    print(f"   ❌ ไม่พบโฟลเดอร์: {len(no_folder)} → {', '.join(r['province'] for r in no_folder)}")

# ตาราง Top 10 + Bottom 10
print(f"\n{'='*70}")
sorted_by_count = sorted(ok_provinces, key=lambda r: r["actual"], reverse=True)
print("📈 Top 10:")
for r in sorted_by_count[:10]:
    print(f"   {r['province']}: {r['actual']:,}")
print("📉 Bottom 10:")
for r in sorted_by_count[-10:]:
    print(f"   {r['province']}: {r['actual']:,}")

# สรุปสุดท้าย
print(f"\n{'='*70}")
coverage = len(ok_provinces) / 77 * 100
print(f"🎯 Coverage: {len(ok_provinces)}/77 จังหวัด ({coverage:.1f}%)")
print(f"📄 Total PDFs: {grand_total:,}")
print(f"📁 {DRIVE_ROOT}")
if len(ok_provinces) >= 75:
    print(f"\n✅ ข้อมูลดิบครบถ้วนสำหรับ {len(ok_provinces)} จังหวัด — พร้อมใช้เป็นหลักฐาน")
    if err_provinces:
        print(f"⚠️ เหลือ {len(err_provinces)} จังหวัดที่ต้องหา folder ID ใหม่: {', '.join(r['province'] for r in err_provinces)}")

*⬆️ ลบ cell นี้ได้ — เป็น cell เก่าที่ไม่ใช้แล้ว*